In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
# This notebook is for a simulation of the protocol without an attacker.

# **Flow for this notebook**

1.   Create Simulator
2.   Bitstring Generator to determine **Alice's Message and Basis** and **Bob's Basis**.
3.   Qubit Generation (Sending and Receiving)
4.   Sifting (To make the private key and bob's measured key for QBER)
5.   QBER for tamper checking
6.   Additional code to see if the average key length is actually half



# **Create Simulator**

In [3]:
simulator = AerSimulator()
n_bits = 16

# **Generate Bitstrings for Alice and Bob**

In [4]:
def generate_bits_quantum(n_bits): # HELPER QUANTUM RANDOMNESS to generate bit and basis

  qc = QuantumCircuit(1, 1)
  qc.h(0)
  qc.measure(0, 0)

  result = simulator.run(qc, shots=n_bits, memory=True).result() # run n_bits times with shots
  bitstring = ''.join(result.get_memory()) # combine together

  return bitstring

# Make Alice and Bob
alice_basis=generate_bits_quantum(n_bits)
alice_state=generate_bits_quantum(n_bits)
bob_basis=generate_bits_quantum(n_bits)

print("Randomly generating Alice and Bob...\n")
print(f"alice_basis: {alice_basis}")
print(f"alice_state: {alice_state}")
print(f"bob_basis: {bob_basis}")

Randomly generating Alice and Bob...

alice_basis: 0111011100000111
alice_state: 0100011010101000
bob_basis: 0000010010011111


# **Function for Alice and Bob to Send and Receive**

In [5]:
"""
LEGEND for BASIS
0 -> standard basis
1 -> diagonal basis

"""

# For Alice, returns the circuit (Qubit)
def sender_function(state, basis):
  bitlength = len(state)

  circuit = QuantumCircuit(bitlength)

  for i in range(bitlength):
    # qubit default is ket 0
    if state[i] == '1': # if bit 1, use pauli-x gate to change to ket 1
      circuit.x(i)

    # after that, check the basis of the bit
    if basis[i] == '1': # if bit 1, use hadamard gate to change into ket - or +
      circuit.h(i)

  return circuit

# For Bob, returns the measured bitstring
def receiver_function(message, measurement_basis):
  bitlength = len(measurement_basis)

  for i in range(bitlength):

    if measurement_basis[i] == '1': # Based on his basis, hadamard gate if 1, else leave it as it is.
      message.h(i)

  # measure to get Bob's state
  message.measure_all()
  result = simulator.run(
        message,
        shots=1,
        memory=True
    ).result()

  return result.get_memory()[0][::-1]

# Simulate Send and Receive
alice_qubit = sender_function(alice_state, alice_basis) # Convert state to qubits with basis
bob_state = receiver_function(alice_qubit, bob_basis) # Convert qubit circuit to state

# **Sifting Function** : Generates the private key and a subset of the receiver's bits based on the basis

In [6]:
# To create private key + receiver key
def sifting_function(sender_basis, sender_state, receiver_basis, receiver_result):
  sender_key = "" # Private Key
  receiver_key = "" # Subset using receiver_state (To calculate QBER)

  for i in range(len(sender_basis)):
    if sender_basis[i] == receiver_basis[i]:
      sender_key += sender_state[i]
      receiver_key += receiver_result[i]

  return sender_key, receiver_key

# Generate Keys
sifted_key, receiver_key = sifting_function(alice_basis, alice_state, bob_basis, bob_state)

# **Check Quantum Bit Error Rate**

In [7]:
# Identify mismatched bits within the matched basis to check for tampering
def calculate_qber(sender_key, receiver_key):

  if len(sender_key) == 0: # if 0
    return 0, 0

  errors = 0

  # count the number of different bits between private key and receiver state
  for i in range(len(sender_key)):
    if sender_key[i] != receiver_key[i]:
      errors += 1

  # length of key divided by number of errors
  qber = errors / len(sender_key)

  return qber, errors

# Calculate QBER
qber, errors = calculate_qber(sifted_key, receiver_key)
print(f"Quantum Bit Error Rate: {qber}")
print(f"Number of errors: {errors}")

Quantum Bit Error Rate: 0.0
Number of errors: 0


# Final Results

In [8]:
# Final Conclusion Block

print("\n==============================")
print("BB84 Protocol Plain Summary")
print("==============================")

print("\n Generated the bits and basis for both Alice and Bob using Quantum Randomness")
print("\n Results as shown: ")
print("----------------------------------")

print("\nAlice basis: ", alice_basis)
print("Alice state: ", alice_state)
print("Bob basis: ", bob_basis)

print("\n Sifting to generate key")
print("----------------------------------")

print("Sifted key (From Alice's Bit): ", sifted_key)
print("Receiver key (From Bob's Measured Bit): ", receiver_key)
print("Sifted key length: ", len(sifted_key))

print("\n QBER Checking to see if message had been tampered")
print("----------------")
print("Number of errors:", errors)
print("QBER:", qber)
print("QBER percentage:", qber * 100, "%")

print("\n Attack Detection")
print("----------------------------------")
if (qber > 0.11): # Following standard of 11% threshold
  print("Attack detected!")
else:
  print("No attack detected.")



BB84 Protocol Plain Summary

 Generated the bits and basis for both Alice and Bob using Quantum Randomness

 Results as shown: 
----------------------------------

Alice basis:  0111011100000111
Alice state:  0100011010101000
Bob basis:  0000010010011111

 Sifting to generate key
----------------------------------
Sifted key (From Alice's Bit):  00101000
Receiver key (From Bob's Measured Bit):  00101000
Sifted key length:  8

 QBER Checking to see if message had been tampered
----------------
Number of errors: 0
QBER: 0.0
QBER percentage: 0.0 %

 Attack Detection
----------------------------------
No attack detected.


Additional to test this part *"This sequence is, on average, half the length of the
sequence of qubits that Alice generates originally."*



In [9]:
def run_bb84(n_bits):
    # Generate Alice and Bob random bits/basis
    alice_basis = generate_bits_quantum(n_bits)
    alice_state = generate_bits_quantum(n_bits)
    bob_basis = generate_bits_quantum(n_bits)

    # Alice sends, Bob receives
    alice_qubit = sender_function(alice_state, alice_basis)
    bob_state = receiver_function(alice_qubit, bob_basis)

    # Sifting
    sifted_key, receiver_key = sifting_function(
        alice_basis,
        alice_state,
        bob_basis,
        bob_state
    )

    # QBER
    qber, errors = calculate_qber(sifted_key, receiver_key)

    return {
        "alice_basis": alice_basis,
        "alice_state": alice_state,
        "bob_basis": bob_basis,
        "bob_state": bob_state,
        "sifted_key": sifted_key,
        "receiver_key": receiver_key,
        "sifted_key_length": len(sifted_key),
        "qber": qber,
        "errors": errors
    }

def test_average_key_length(n_bits, trials):
    key_lengths = []

    for i in range(trials):
        result = run_bb84(n_bits)
        key_lengths.append(result["sifted_key_length"])

    average_key_length = sum(key_lengths) / trials
    expected_key_length = n_bits / 2

    print("\n==============================")
    print("BB84 Average Key Length Test")
    print("==============================")

    print(f"Number of bits per run: {n_bits}")
    print(f"Number of trials: {trials}")

    print("\nResults")
    print("----------------------------------")
    print(f"Average sifted key length: {average_key_length}")
    print(f"Expected key length: {expected_key_length}")
    print(f"Average percentage kept: {(average_key_length / n_bits) * 100:.2f}%")

    print("\nTheory Check")
    print("----------------------------------")
    if abs(average_key_length - expected_key_length) < 1:
        print("Result supports the theory: the sifted key is approximately half the original bit length.")
    else:
        print("Result is not close yet. Try increasing the number of trials.")

    return key_lengths

In [10]:
test_average_key_length(16, 50)


BB84 Average Key Length Test
Number of bits per run: 16
Number of trials: 50

Results
----------------------------------
Average sifted key length: 7.94
Expected key length: 8.0
Average percentage kept: 49.62%

Theory Check
----------------------------------
Result supports the theory: the sifted key is approximately half the original bit length.


[4,
 9,
 7,
 8,
 8,
 6,
 9,
 13,
 5,
 10,
 8,
 9,
 9,
 9,
 5,
 9,
 8,
 8,
 8,
 9,
 8,
 9,
 8,
 7,
 6,
 10,
 8,
 6,
 10,
 8,
 6,
 9,
 9,
 8,
 9,
 8,
 9,
 9,
 6,
 8,
 7,
 8,
 5,
 11,
 8,
 8,
 6,
 5,
 7,
 10]